<a href="https://colab.research.google.com/github/kjm5416-ship-it/secom-anomaly-detection/blob/main/04_%EB%AA%A8%EB%8D%B85%EC%A2%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np

path = '/content/drive/MyDrive/도전학기'
X_train = pd.read_csv(f'{path}/X_train.csv')
y_train = pd.read_csv(f'{path}/y_train.csv').squeeze()
X_test  = pd.read_csv(f'{path}/X_test.csv')
y_test  = pd.read_csv(f'{path}/y_test.csv').squeeze()

print("학습용:", X_train.shape, "| 불량", y_train.sum(), "개")
print("시험용:", X_test.shape, "| 불량", y_test.sum(), "개")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
학습용: (1253, 446) | 불량 83 개
시험용: (314, 446) | 불량 21 개


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (confusion_matrix, recall_score, precision_score,
                             f1_score, average_precision_score, roc_auc_score)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def 채점(name, model):
    pred = cross_val_predict(model, X_train, y_train, cv=cv, method='predict')
    prob = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()

    return {
        '방법': name,
        '잡은 불량': tp,
        '놓친 불량': fn,
        '헛알람': fp,
        'Recall':    recall_score(y_train, pred, zero_division=0),
        'Precision': precision_score(y_train, pred, zero_division=0),
        'F1':        f1_score(y_train, pred, zero_division=0),
        'PR-AUC':    average_precision_score(y_train, prob),
        'ROC-AUC':   roc_auc_score(y_train, prob),
    }

print("채점 함수 준비 완료")

채점 함수 준비 완료


In [ ]:
from sklearn.linear_model import LogisticRegression

결과 = []
결과.append(채점('로지스틱 회귀', LogisticRegression(max_iter=2000, random_state=42)))

pd.set_option('display.width', 200)
pd.DataFrame(결과).round(3)

,방법,잡은 불량,놓친 불량,헛알람,Recall,Precision,F1,PR-AUC,ROC-AUC
0,로지스틱 회귀,17,66,90,0.205,0.159,0.179,0.123,0.618


In [ ]:
결과.append(채점('로지스틱 (가중치 보정)',
                LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)))
pd.DataFrame(결과).round(3)

,방법,잡은 불량,놓친 불량,헛알람,Recall,Precision,F1,PR-AUC,ROC-AUC
0,로지스틱 회귀,17,66,90,0.205,0.159,0.179,0.123,0.618
1,로지스틱 (가중치 보정),20,63,129,0.241,0.134,0.172,0.123,0.616


In [ ]:
from sklearn.base import clone

def 채점_이상탐지(name, model):
    pred  = np.zeros(len(y_train), dtype=int)
    score = np.zeros(len(y_train))

    for tr, te in cv.split(X_train, y_train):
        m = clone(model)
        m.fit(X_train.iloc[tr])                              # 라벨을 주지 않는다
        pred[te]  = (m.predict(X_train.iloc[te]) == -1).astype(int)
        score[te] = -m.decision_function(X_train.iloc[te])

    tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()
    return {'방법': name, '잡은 불량': tp, '놓친 불량': fn, '헛알람': fp,
            'Recall':    recall_score(y_train, pred, zero_division=0),
            'Precision': precision_score(y_train, pred, zero_division=0),
            'F1':        f1_score(y_train, pred, zero_division=0),
            'PR-AUC':    average_precision_score(y_train, score),
            'ROC-AUC':   roc_auc_score(y_train, score)}

print("이상탐지 채점 함수 준비 완료")

이상탐지 채점 함수 준비 완료


In [ ]:
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from xgboost import XGBClassifier

RATE = y_train.mean()   # 0.0662

결과 = []
결과.append(채점('로지스틱 회귀',
    LogisticRegression(max_iter=2000, random_state=42)))
결과.append(채점('랜덤 포레스트',
    RandomForestClassifier(n_estimators=300, class_weight='balanced',
                           random_state=42, n_jobs=-1)))
결과.append(채점('XGBoost',
    XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                  scale_pos_weight=(1-RATE)/RATE,
                  eval_metric='logloss', random_state=42, n_jobs=-1)))
결과.append(채점_이상탐지('Isolation Forest',
    IsolationForest(n_estimators=300, contamination=RATE,
                    random_state=42, n_jobs=-1)))
결과.append(채점_이상탐지('One-Class SVM',
    OneClassSVM(kernel='rbf', gamma='scale', nu=RATE)))

비교표 = pd.DataFrame(결과)
pd.set_option('display.width', 220)
비교표.round(3)

,방법,잡은 불량,놓친 불량,헛알람,Recall,Precision,F1,PR-AUC,ROC-AUC
0,로지스틱 회귀,17,66,90,0.205,0.159,0.179,0.123,0.618
1,랜덤 포레스트,0,83,0,0.000,0.000,0.000,0.178,0.713
2,XGBoost,2,81,10,0.024,0.167,0.042,0.150,0.663
3,Isolation Forest,10,73,73,0.120,0.120,0.120,0.104,0.542
4,One-Class SVM,17,66,113,0.205,0.131,0.160,0.091,0.546


In [ ]:
비교표.to_csv(f'{path}/성능비교표_v1.csv', index=False)
print("저장 완료")

저장 완료
